# Python Data Types — Interactive Notebook

Hands-on exploration of CPython internals, memory layout, and performance characteristics.

## 1. Memory Sizes with sys.getsizeof()

In [ ]:
import sys

types_and_values = [
    ("int(0)",        0),
    ("int(256)",      256),
    ("int(257)",      257),
    ("int(2**30)",    2**30),
    ("int(2**60)",    2**60),
    ("float",         3.14),
    ("bool(True)",    True),
    ("str('')",       ""),
    ("str('hello')",  "hello"),
    ("str(emoji)",    "\U0001F600"),
    ("bytes(5)",      b"hello"),
    ("bytearray(5)",  bytearray(5)),
    ("list([])",      []),
    ("list([1..5])",  [1, 2, 3, 4, 5]),
    ("tuple(())",     ()),
    ("tuple(1..5)",   (1, 2, 3, 4, 5)),
    ("dict({})",      {}),
    ("dict(1 item)",  {"a": 1}),
    ("set(empty)",    set()),
    ("frozenset",     frozenset([1, 2, 3])),
    ("None",          None),
]

print(f"{'Type/Value':<22} {'Size (bytes)':>12}")
print("-" * 36)
for name, val in types_and_values:
    print(f"{name:<22} {sys.getsizeof(val):>12}")

## 2. Integer Caching Demo (Small Int Cache: -5 to 256)

In [ ]:
print("=== Small Integer Cache (-5 to 256) ===")
print()

# Cached integers — same object
for val in [-5, 0, 1, 100, 256]:
    a = val
    b = val
    print(f"val={val:4d}: a is b = {a is b}, id(a)={id(a)}, id(b)={id(b)}")

print()
print("=== Outside Cache (257+) ===")
print()

# Outside cache — different objects (in most contexts)
for val in [257, 1000, -6]:
    a = val
    b = val
    print(f"val={val:4d}: a is b = {a is b}, id(a)={id(a)}, id(b)={id(b)}")

print()
print("Note: In interactive REPL or same code block, CPython may still reuse")
print("objects for literals due to constant folding. True difference shows in")
print("separate function calls or exec().")

In [ ]:
# Demonstrate cache boundary clearly using exec
import ctypes

def make_int(n):
    """Create integer in a separate scope to avoid constant folding."""
    return n

a = make_int(256)
b = make_int(256)
print(f"256: a is b = {a is b}")  # True — cached

a = make_int(257)
b = make_int(257)
print(f"257: a is b = {a is b}")  # False — not cached

# Show memory savings from cache
print(f"\nMemory for int(0):   {sys.getsizeof(0)} bytes")
print(f"Memory for int(256): {sys.getsizeof(256)} bytes")
print(f"Memory for int(257): {sys.getsizeof(257)} bytes")
print(f"Memory for int(2**30): {sys.getsizeof(2**30)} bytes  (needs 2 digits)")
print(f"Memory for int(2**60): {sys.getsizeof(2**60)} bytes  (needs 3 digits)")
print(f"Memory for int(2**90): {sys.getsizeof(2**90)} bytes  (needs 4 digits)")

## 3. Float Precision Issues and Fixes

In [ ]:
import struct
import math
from decimal import Decimal, getcontext
from fractions import Fraction

print("=== The Classic Problem ===")
print(f"0.1 + 0.2 = {0.1 + 0.2}")
print(f"0.1 + 0.2 == 0.3: {0.1 + 0.2 == 0.3}")
print()

print("=== IEEE 754 Representation ===")
for val in [0.1, 0.2, 0.3]:
    bits = struct.pack('>d', val)
    print(f"{val} -> hex bytes: {bits.hex()} -> repr: {repr(val)}")
print()

print("=== Fix 1: math.isclose ===")
print(f"math.isclose(0.1+0.2, 0.3) = {math.isclose(0.1 + 0.2, 0.3)}")
print()

print("=== Fix 2: decimal.Decimal ===")
getcontext().prec = 50
d1 = Decimal('0.1')
d2 = Decimal('0.2')
d3 = Decimal('0.3')
print(f"Decimal('0.1') + Decimal('0.2') = {d1 + d2}")
print(f"== Decimal('0.3'): {d1 + d2 == d3}")
print()

print("=== Fix 3: fractions.Fraction ===")
f1 = Fraction(1, 10)
f2 = Fraction(2, 10)
print(f"Fraction(1,10) + Fraction(2,10) = {f1 + f2}")
print(f"== Fraction(3,10): {f1 + f2 == Fraction(3, 10)}")
print()

print("=== Special Float Values ===")
print(f"float('inf')  = {float('inf')}")
print(f"float('-inf') = {float('-inf')}")
print(f"float('nan')  = {float('nan')}")
print(f"nan == nan: {float('nan') == float('nan')}  (NaN is never equal to itself!)")
print(f"math.isnan(nan): {math.isnan(float('nan'))}")

## 4. List Over-Allocation Demo

In [ ]:
import sys
import ctypes

print("=== List Growth Pattern ===")
print(f"{'len':>5} {'sys.getsizeof':>15} {'allocated (est)':>16}")
print("-" * 40)

lst = []
prev_size = sys.getsizeof(lst)
print(f"{0:>5} {prev_size:>15} {0:>16}")

for i in range(1, 33):
    lst.append(i)
    size = sys.getsizeof(lst)
    # Estimate allocated slots: (size - 56) // 8  (56 = base list size, 8 = ptr size)
    allocated = (size - 56) // 8
    if size != prev_size:
        print(f"{i:>5} {size:>15} {allocated:>16}  <-- reallocation")
        prev_size = size
    else:
        print(f"{i:>5} {size:>15} {allocated:>16}")

In [ ]:
# Visualize growth pattern
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

sizes = []
lst = []
for i in range(50):
    lst.append(i)
    sizes.append(sys.getsizeof(lst))

plt.figure(figsize=(10, 4))
plt.step(range(1, 51), sizes, where='post', color='steelblue', linewidth=2)
plt.xlabel('Number of elements')
plt.ylabel('sys.getsizeof (bytes)')
plt.title('Python List Memory Growth (Over-Allocation)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('list_growth.png', dpi=100)
plt.show()
print("Chart saved as list_growth.png")

## 5. Dict Hash Collision Demo

In [ ]:
print("=== Hash Values ===")
keys = [0, 1, 2, 'a', 'b', 'hello', (1, 2), 3.0, True]
for k in keys:
    print(f"hash({repr(k):>12}) = {hash(k)}")

print()
print("=== Interesting: Equal objects have equal hashes ===")
print(f"hash(1)    = {hash(1)}")
print(f"hash(1.0)  = {hash(1.0)}")
print(f"hash(True) = {hash(True)}")
print(f"1 == 1.0 == True: {1 == 1.0 == True}")
print()

print("=== Consequence: dict treats them as same key ===")
d = {}
d[1]    = 'int one'
d[1.0]  = 'float one'   # overwrites d[1]!
d[True] = 'bool True'   # overwrites again!
print(f"d = {d}")
print(f"len(d) = {len(d)}  (only 1 key!)")

In [ ]:
print("=== Dict Ordering (Python 3.7+) ===")
d = {}
for char in 'python':
    d[char] = ord(char)
print(f"Insertion order preserved: {list(d.keys())}")

print()
print("=== Dict Memory Growth ===")
d = {}
prev = sys.getsizeof(d)
print(f"empty dict: {prev} bytes")
for i in range(10):
    d[i] = i
    size = sys.getsizeof(d)
    if size != prev:
        print(f"After {len(d):2d} items: {size} bytes  <-- resize")
        prev = size
    else:
        print(f"After {len(d):2d} items: {size} bytes")

## 6. String Interning Demo

In [ ]:
import sys

print("=== Automatic Interning (identifier-like strings) ===")
a = "hello"
b = "hello"
print(f"'hello' is 'hello': {a is b}  (interned)")

a = "hello world"  # has space — not auto-interned
b = "hello world"
print(f"'hello world' is 'hello world': {a is b}  (may vary by context)")

print()
print("=== Manual Interning ===")
a = sys.intern("hello world")
b = sys.intern("hello world")
print(f"After sys.intern: {a is b}  (True — same object)")

print()
print("=== String Memory by Content Type ===")
s_ascii  = "hello"          # Latin-1: 1 byte/char
s_latin  = "caf\u00e9"      # Latin-1: 1 byte/char (é = U+00E9)
s_ucs2   = "\u4e2d\u6587"   # UCS-2: 2 bytes/char (Chinese)
s_ucs4   = "\U0001F600"     # UCS-4: 4 bytes/char (emoji)

for name, s in [("ASCII 'hello'", s_ascii), ("Latin-1 'café'", s_latin),
                ("UCS-2 Chinese", s_ucs2), ("UCS-4 Emoji", s_ucs4)]:
    print(f"{name:20s}: {sys.getsizeof(s):3d} bytes, len={len(s)}")

## 7. Benchmark: list vs tuple Creation

In [ ]:
import timeit

N = 5_000_000

t_tuple = timeit.timeit("(1, 2, 3, 4, 5)", number=N)
t_list  = timeit.timeit("[1, 2, 3, 4, 5]", number=N)

print(f"Tuple creation ({N:,} times): {t_tuple:.3f}s")
print(f"List  creation ({N:,} times): {t_list:.3f}s")
print(f"Tuple is {t_list/t_tuple:.1f}x faster than list")

print()
# Iteration benchmark
setup = "data = list(range(1000)); tdata = tuple(range(1000))"
t_list_iter  = timeit.timeit("for x in data: pass",  setup=setup, number=100_000)
t_tuple_iter = timeit.timeit("for x in tdata: pass", setup=setup, number=100_000)

print(f"List  iteration (100k × 1000 items): {t_list_iter:.3f}s")
print(f"Tuple iteration (100k × 1000 items): {t_tuple_iter:.3f}s")
print(f"Tuple is {t_list_iter/t_tuple_iter:.2f}x faster for iteration")

## 8. Set vs List Membership Test

In [ ]:
import timeit

setup = """
lst = list(range(100_000))
st  = set(range(100_000))
"""

t_list = timeit.timeit("99_999 in lst", setup=setup, number=10_000)
t_set  = timeit.timeit("99_999 in st",  setup=setup, number=10_000)

print(f"List membership (worst case, 10k lookups): {t_list*1000:.2f} ms")
print(f"Set  membership (worst case, 10k lookups): {t_set*1000:.4f} ms")
print(f"Set is {t_list/t_set:.0f}x faster than list for membership test")

## 9. Mutability Demo

In [ ]:
print("=== Mutable types can be changed in-place ===")
lst = [1, 2, 3]
print(f"Before: id={id(lst)}, lst={lst}")
lst.append(4)
print(f"After:  id={id(lst)}, lst={lst}  (same object!)")

print()
print("=== Immutable types create new objects ===")
s = "hello"
print(f"Before: id={id(s)}, s={s!r}")
s = s + " world"
print(f"After:  id={id(s)}, s={s!r}  (new object!)")

print()
print("=== Tuple with mutable element ===")
t = ([1, 2], [3, 4])
print(f"Before: {t}")
t[0].append(99)  # modifying the list inside the tuple is OK
print(f"After:  {t}  (tuple unchanged, inner list mutated)")
# t[0] = [99]  # TypeError — can't rebind tuple element

## 10. Type Conversion Gotchas

In [ ]:
print("=== int() truncates, does NOT round ===")
print(f"int(3.9)  = {int(3.9)}   (truncated to 3, not 4)")
print(f"int(-3.9) = {int(-3.9)}  (truncated toward zero)")
print(f"round(3.9) = {round(3.9)}  (rounded to 4)")

print()
print("=== int() with base ===")
print(f"int('0xFF', 16) = {int('0xFF', 16)}")
print(f"int('0b1010', 2) = {int('0b1010', 2)}")
print(f"int('0o17', 8) = {int('0o17', 8)}")

print()
print("=== bool() truthiness ===")
falsy = [0, 0.0, 0j, '', [], (), {}, set(), None, False]
print("Falsy values:", [repr(x) for x in falsy])
print("All bool() False:", all(not bool(x) for x in falsy))

print()
print("=== Surprising: bool is int ===")
print(f"True + True = {True + True}")
print(f"sum([True, False, True, True]) = {sum([True, False, True, True])}")
print(f"True * 42 = {True * 42}")
print(f"isinstance(True, int) = {isinstance(True, int)}")

## 11. Interview Q Demos

In [ ]:
# Q1: Why 0.1 + 0.2 != 0.3?
import struct
print("Q1: Float precision")
for v in [0.1, 0.2, 0.3, 0.1+0.2]:
    bits = struct.pack('>d', v)
    print(f"  {v} -> {bits.hex()} (exact stored value: {v:.20f})")

In [ ]:
# Q3: List over-allocation amortized O(1)
print("Q3: Amortized O(1) append")
import sys
lst = []
reallocs = 0
prev_size = sys.getsizeof(lst)
for i in range(1000):
    lst.append(i)
    size = sys.getsizeof(lst)
    if size != prev_size:
        reallocs += 1
        prev_size = size
print(f"  1000 appends triggered {reallocs} reallocations")
print(f"  Average: 1 reallocation per {1000/reallocs:.0f} appends")

In [ ]:
# Q4: Dict hash collision
print("Q4: Hash collision — 1, 1.0, True all hash the same")
print(f"  hash(1)    = {hash(1)}")
print(f"  hash(1.0)  = {hash(1.0)}")
print(f"  hash(True) = {hash(True)}")
d = {1: 'int', 1.0: 'float', True: 'bool'}
print(f"  dict with keys 1, 1.0, True: {d}  (only 1 entry!)")

In [ ]:
# Q5: Tuple vs list speed
import timeit
print("Q5: Tuple vs list creation speed")
t = timeit.timeit("(1,2,3,4,5)", number=10_000_000)
l = timeit.timeit("[1,2,3,4,5]", number=10_000_000)
print(f"  Tuple: {t:.3f}s")
print(f"  List:  {l:.3f}s")
print(f"  Speedup: {l/t:.1f}x")

# Disassemble to see why
import dis
print()
print("  Bytecode for tuple literal:")
dis.dis(compile("(1,2,3)", "", "eval"))
print()
print("  Bytecode for list literal:")
dis.dis(compile("[1,2,3]", "", "eval"))